## Pattern {PATTERN_NAME} Retrieve & Generation Content

This notebook demonstrates how to implement and test a Retrieval-Augmented Generation (RAG) pattern using OGX. It guides you through setting up the necessary components, loading test data from an S3 bucket, and querying the RAG system to generate responses based on retrieved context.

&#x26A0;&#xFE0F; **Important**: Before running this notebook, you must first run the corresponding **indexing.ipynb** notebook to populate the vector store with document embeddings. The indexing process prepares the knowledge base that this notebook queries.

### &#x1F4CB; Contents 
This notebook contains the following sections:

- **[Setup](#Setup)**
- **[Prepare OgxClient](#Prepare-OgxClient)**
- **[Initialize RAG Components](#Initialize-RAG-Components)**
   - [Initialize OGX Foundation Model](#Initialize-OGX-Foundation-Model)
   - [Initialize Vector Store Client](#Initialize-Vector-Store-Client)
   - [Initialize Retriever](#Initialize-Retriever)
   - [Query RAG Pattern](#Query-RAG-Pattern)
- **[Next steps](#Next-steps)**
   - [Load Test Data](#Load-Test-Data)
   - [Configure S3 Credentials](#Configure-S3-Credentials)
   - [Initialize S3 Client](#Initialize-S3-Client)
   - [Load Benchmark Data](#Load-Benchmark-Data)
   - [Build Evaluation Data](#Build-Evaluation-Data)
   - [Evaluate Response](#Evaluate-Response)
- **[Summary](#Summary)**

---

## Setup

This section installs all the required Python packages for running the RAG experiment:
- **boto3**: AWS SDK for Python to interact with S3 storage
- **pipelines-components**: Red Hat's pipeline components for data processing
- **ai4rag**: The main RAG framework for AutoRAG experiments

In [ ]:
%pip install boto3 | tail -n 1
%pip install -U --no-cache-dir git+https://github.com/red-hat-data-services/pipelines-components.git@rhoai-3.4 | tail -n 1
%pip install 'ai4rag~=0.6.1' | tail -n 1

---

## Prepare OgxClient

The OGX client is the core interface for interacting with the OGX API. This section initializes the client by:
- Retrieving API credentials from environment variables or prompting for them
- Establishing a connection to the OGX endpoint

**Prerequisites:**
- `OGX_CLIENT_API_KEY`: Your authentication key for the OGX API
- `OGX_CLIENT_BASE_URL`: The base URL of your OGX instance

&#x1F4A1; **Tip**: In OpenShift AI Workbench, you can add these as environment variables or data connections to avoid entering them manually each time.

In [ ]:
import getpass
import logging
import os
import warnings

import httpx
from ogx_client import OgxClient

warnings.filterwarnings("ignore")
logging.getLogger("httpx").propagate = False

if not os.getenv("OGX_CLIENT_API_KEY") or not os.getenv("OGX_CLIENT_BASE_URL"):
    os.environ["OGX_CLIENT_API_KEY"] = getpass.getpass("Please enter 'OGX_CLIENT_API_KEY': ")
    os.environ["OGX_CLIENT_BASE_URL"] = getpass.getpass("Please enter 'OGX_CLIENT_BASE_URL': ")

_ls_kwargs = dict(
    api_key=os.getenv("OGX_CLIENT_API_KEY"),
    base_url=os.getenv("OGX_CLIENT_BASE_URL"),
)
if not {LS_SSL_VERIFY}:
    _ls_kwargs["http_client"] = httpx.Client(verify=False)

client = OgxClient(**_ls_kwargs)

---

## Initialize RAG Components

This section sets up all the components needed for the RAG pattern: foundation model, vector store, retriever, and the RAG pattern itself.

### Initialize OGX Foundation Model

The foundation model is the core language model that generates responses. This section configures:
- **Model ID**: The specific OGX model to use for generation
- **System Message**: Instructions that define the model's behavior and role
- **User Message Template**: The format for user queries
- **Context Template**: How retrieved context is incorporated into prompts

These templates control how the RAG system structures prompts to the language model.

In [ ]:
from ai4rag.rag.foundation_models.ogx import OGXFoundationModel

chat_model_id = """{FM_MODEL_ID}"""
system_message_text = """{SYSTEM_MESSAGE}"""
user_message_text = """{USER_MESSAGE}"""
context_template_text = """{CONTEXT_TEXT}"""

ogx_foundation_model = OGXFoundationModel(
    client=client,
    model_id=chat_model_id,
    system_message_text=system_message_text,
    user_message_text=user_message_text,
    context_template_text=context_template_text,
)

### Initialize Vector Store Client

The vector store is responsible for storing and retrieving document embeddings. This section sets up:
- **Embedding Model**: Converts text into vector representations for semantic search
- **Vector Database**: Stores embeddings with configurable distance metrics (cosine, euclidean, etc.)
- **Collection**: A named collection where document vectors are stored and can be reused

The vector store enables semantic similarity search to find relevant context for user queries.

In [ ]:
from ai4rag.rag.embedding.ogx import OGXEmbeddingModel, OGXEmbeddingParams
from ai4rag.rag.vector_store.ogx import OGXVectorStore

embedding_model_id = "{EMBEDDING_MODEL_ID}"
params = OGXEmbeddingParams(**{EMBEDDING_PARAMS})
embedding_model = OGXEmbeddingModel(client=client, model_id=embedding_model_id, params=params)
provider_id = "{PROVIDER_ID}"
distance_metric = "{DISTANCE_METRIC}"
collection_name = "{COLLECTION_NAME}"

ogx_vector_store = OGXVectorStore(
    embedding_model=embedding_model,
    client=client,
    provider_id=provider_id,
    distance_metric=distance_metric,
    reuse_collection_name=collection_name,
)

### Initialize Retriever

The retriever finds the most relevant document chunks for a given query. Configuration includes:
- **Retrieval Method**: The algorithm used to find relevant documents (e.g., similarity search, hybrid search)
- **Number of Chunks**: How many document chunks to retrieve and include in the context

The retriever acts as the bridge between user questions and the knowledge base.

In [ ]:
from ai4rag.rag.retrieval.retriever import Retriever

method = "{RETRIEVAL_METHOD}"
number_of_chunks = {NUMBER_OF_CHUNKS}
search_mode = "{SEARCH_MODE}"
ranker_strategy = "{RANKER_STRATEGY}"
ranker_k = {RANKER_K}
ranker_alpha = {RANKER_ALPHA}

# Build retriever kwargs, only including non-None hybrid search parameters
retriever_kwargs = dict(
    (("vector_store", ogx_vector_store),
    ("method", method),
    ("number_of_chunks", number_of_chunks))
)
if search_mode is not None:
    retriever_kwargs["search_mode"] = search_mode
if ranker_strategy != "None":
    retriever_kwargs["ranker_strategy"] = ranker_strategy
if ranker_k is not None:
    retriever_kwargs["ranker_k"] = ranker_k
if ranker_alpha is not None:
    retriever_kwargs["ranker_alpha"] = ranker_alpha

retriever = Retriever(**retriever_kwargs)

---

### Query RAG Pattern

This section executes the RAG workflow by submitting test questions to the system and generating responses based on retrieved context.

In [ ]:
from ai4rag.rag.template.simple_rag_template import SimpleRAG

rag_pattern = SimpleRAG(foundation_model=ogx_foundation_model, retriever=retriever)

In [ ]:
from pprint import pprint

question = input()
response = rag_pattern.generate(question=question)
pprint(response, indent=4, width=50)

---

## Next steps

The following sections provide optional next steps for loading test data, running queries, and evaluating the RAG pattern's performance. These steps are useful for systematic testing and benchmarking, but can be skipped if you prefer to interact with the RAG system directly using the pattern configured above.

---

### Load Test Data

This section prepares the test environment and loads benchmark questions from S3 storage. The test data is used to evaluate the RAG system's performance.

```python
import os
import json
from pathlib import Path
from types import SimpleNamespace

import boto3

logging.getLogger('Test Data Loader component logger').propagate = False
```

### Configure S3 Credentials

To load test data from S3-compatible object storage, you need to provide credentials. If you're using OpenShift AI, these can be configured as data connections.

&#x1F4CC; **Action**: Provide the credentials for your S3 instance if they are not already set in the notebook environment.

&#x1F4A1; **Tip**: In the project, open **Connections** and add an **S3 compatible object storage connection** to a bucket you will use for documents and test data. Open **Workbenches**, edit your workbench, and attach the S3 connection you created so the notebook can read from the bucket. Save and restart the workbench if prompted.

```python
required_vars = ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_S3_ENDPOINT", "AWS_DEFAULT_REGION", "AWS_S3_BUCKET"]
missing = [var for var in required_vars if not os.environ.get(var)]
if missing:
    raise ValueError(f"Missing required environment variables: {{missing}}")
```

### Initialize S3 Client

Creates an S3 client session using the provided credentials. This client is used to download test data from the specified S3 bucket.

```python
session = boto3.session.Session(
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
)
s3_client = session.client(
    service_name='s3',
    endpoint_url=os.environ["AWS_S3_ENDPOINT"],
)
```

### Load Benchmark Data

Downloads and loads the benchmark test data from S3. The benchmark file should be a JSON file containing:
- **question**: The test question to ask the RAG system
- **correct_answers**: The expected answers for evaluation
- **correct_answer_document_ids**: IDs of documents that contain the correct information

This data is used to measure the RAG system's accuracy and retrieval quality.

```python
from kfp_components.components.data_processing.autorag.test_data_loader.component import test_data_loader


step_output_dir = Path("./step_outputs")
step_output_dir.mkdir(parents=True, exist_ok=True)

test_data_bucket_name = os.environ['AWS_S3_BUCKET']
test_data_key = "{TEST_DATA_KEY}"
test_data_out = SimpleNamespace(path=str(step_output_dir / "test_data.json"))

test_data_loader.python_func(
    test_data_bucket_name=test_data_bucket_name,
    test_data_path=test_data_key,
    test_data=test_data_out,
)

output_path = Path(test_data_out.path)
with output_path.open("r", encoding="utf-8") as f:
    test_data = json.load(f)

print(json.dumps(test_data, indent=4, ensure_ascii=False))
```

```python
inference_responses = []

for test_data_item in test_data:
    response = rag_pattern.generate(question=test_data_item["question"])
    inference_responses.append(response)
```

### Build Evaluation Data

This section transforms the RAG system's inference responses into a structured format for evaluation. It combines:
- **Benchmark Data**: The original test questions and expected answers
- **Inference Responses**: The actual responses generated by the RAG system

The resulting evaluation data structure allows for systematic comparison between expected and actual outputs, enabling metric calculation for assessing the RAG system's performance.

```python
from pandas import DataFrame

from ai4rag.core.experiment.utils import build_evaluation_data
from ai4rag.core.experiment.benchmark_data import BenchmarkData


benchmark_data = BenchmarkData(
    DataFrame(
        data=test_data
    )
)

evaluation_data = build_evaluation_data(
    benchmark_data=benchmark_data, 
    inference_response=inference_responses
)
```

### Evaluate Response

This section evaluates the quality of the RAG system's responses by comparing them against the expected answers from the benchmark data. Evaluation metrics may include accuracy, relevance, and retrieval precision.

```python
from ai4rag.evaluator.unitxt_evaluator import UnitxtEvaluator
from ai4rag.evaluator.base_evaluator import MetricType

evaluator = UnitxtEvaluator()
evaluator.evaluate_metrics(evaluation_data=evaluation_data, metrics=(MetricType.ANSWER_CORRECTNESS, MetricType.FAITHFULNESS, MetricType.CONTEXT_CORRECTNESS))
```

---

## Summary

This notebook successfully demonstrates a complete RAG pattern implementation using OGX, from initializing the foundation model and vector store to querying the system with test data. The evaluation framework allows you to measure the quality of generated responses against benchmark answers using multiple metrics including answer correctness, faithfulness, and context correctness.